In [0]:
%sql
-- 1. A folder for your tables
CREATE SCHEMA IF NOT EXISTS workspace.first_project_autoload;

-- 2. A folder for your files
CREATE VOLUME IF NOT EXISTS workspace.default.book_data;

-- 3. Then upload the CSV using the Catalog menu on the left:
--    Catalog > workspace > default > book_data > Upload to this volume
--    Upload first_project_orders.csv

--4. A folder to save your schema for streaming
CREATE VOLUME IF NOT EXISTS workspace.default.schema;

--5. A folder to save your checkpoint for streaming
CREATE VOLUME IF NOT EXISTS workspace.default.checkpoints;

--6 creating a silver table with expected schema
CREATE TABLE IF NOT EXISTS workspace.first_project_autoload.orders_silver (
    order_id STRING,
    customer_id STRING,
    product STRING,
    region STRING,
    quantity INT,
    amount DOUBLE,
    order_date DATE,
    status STRING
)
USING DELTA;


In [0]:
from pyspark.sql import functions as F

bronze_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/default/schema/orders/")
    .load("/Volumes/workspace/default/book_data/")
)

bronze_stream = (
    bronze_stream
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("loaded_at", F.current_timestamp())
)

query = (
    bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/workspace/default/checkpoints/bronze/orders/")
    .trigger(availableNow=True)
    .toTable("workspace.first_project_autoload.orders_bronze")
)

query.awaitTermination()

In [0]:
%sql
select * from workspace.first_project_autoload.orders_bronze order by order_id;

order_id,customer_id,product,region,quantity,amount,order_date,status,_rescued_data,source_file,loaded_at
ORD-1001,CUST-119,Headset,North,1,741.83,2026-08-15,completed,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T09:47:35.576Z
ORD-1002,CUST-174,Notebook,South,1,91.06,2026-08-15,returned,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T09:47:35.576Z
ORD-1003,CUST-130,Cable,West,1,821.44,2026-08-11,completed,null,/Volumes/workspace/default/book_data/first_project_orders_updates.csv,2026-08-21T09:57:33.496Z
ORD-1003,CUST-130,Cable,West,1,746.76,2026-08-11,completed,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T09:47:35.576Z
ORD-1004,CUST-180,Notebook,West,1,878.99,2026-08-13,completed,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T09:47:35.576Z
ORD-1004,CUST-180,Notebook,West,1,878.99,2026-08-13,completed,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T09:47:35.576Z
ORD-1005,CUST-117,Mouse,West,2,493.51,2026-08-18,pending,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T09:47:35.576Z
ORD-1006,CUST-171,Monitor,North,5,520.52,2026-08-14,completed,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T09:47:35.576Z
ORD-1007,CUST-112,Cable,North,5,197.27,2026-08-15,pending,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T09:47:35.576Z
ORD-1008,CUST-140,Webcam,West,3,not_available,2026-08-16,completed,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T09:47:35.576Z


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

def process_batch(batch_df, batch_id):
    typed_df = batch_df.withColumns({
        "region": F.initcap(F.trim(F.col("region"))),
        "amount": F.expr("try_cast(amount AS DOUBLE)"),
        "order_date": F.expr("try_cast(order_date AS DATE)")
    })

    is_valid = (
        F.col("amount").isNotNull()
        & (F.col("amount") >= 0)
        & F.col("order_date").isNotNull()
    )

    clean_df = typed_df.filter(is_valid).select(
                    "order_id",
                    "customer_id",
                    "product",
                    "region",
                    "quantity",
                    "amount",
                    "order_date",
                    "status"
                ).dropDuplicates(["order_id"])
    rejected_df = batch_df.join(
        clean_df,
        on="order_id",
        how="left_anti"
    )

    silver_table = DeltaTable.forName(spark, "workspace.first_project_autoload.orders_silver")

    (
        silver_table.alias("target").merge(
            clean_df.alias("source"),
            "target.order_id = source.order_id"
        ).whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    rejected_df.write.mode("append").saveAsTable(
        "workspace.first_project_autoload.orders_rejected"
    )

silver_source = spark.readStream.table(
    "workspace.first_project_autoload.orders_bronze"
)

query = (
    silver_source.writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", "/Volumes/workspace/default/checkpoints/silver/orders/")
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
%sql
Select * from first_project_autoload.orders_silver order by order_id;

order_id,customer_id,product,region,quantity,amount,order_date,status
ORD-1001,CUST-119,Headset,North,1,741.83,2026-08-15,completed
ORD-1002,CUST-174,Notebook,South,1,91.06,2026-08-15,returned
ORD-1003,CUST-130,Cable,West,1,746.76,2026-08-11,completed
ORD-1004,CUST-180,Notebook,West,1,878.99,2026-08-13,completed
ORD-1005,CUST-117,Mouse,West,2,493.51,2026-08-18,pending
ORD-1006,CUST-171,Monitor,North,5,520.52,2026-08-14,completed
ORD-1007,CUST-112,Cable,North,5,197.27,2026-08-15,pending
ORD-1009,CUST-110,Mouse,West,3,660.56,2026-08-13,completed
ORD-1010,CUST-109,Cable,West,2,685.07,2026-08-19,completed
ORD-1011,CUST-153,Notebook,North,5,522.13,2026-08-17,completed


In [0]:
%sql
Select * from first_project_autoload.orders_rejected order by order_id;

order_id,customer_id,product,region,quantity,amount,order_date,status,_rescued_data,source_file,loaded_at
ORD-1008,CUST-140,Webcam,West,3,not_available,2026-08-16,completed,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T08:36:07.974Z
ORD-1008,CUST-140,Webcam,West,3,not_available,2026-08-16,completed,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T09:47:35.576Z
ORD-1023,CUST-176,Notebook,North,1,516.6,08/13/2026,pending,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T09:47:35.576Z
ORD-1023,CUST-176,Notebook,North,1,516.6,08/13/2026,pending,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T08:36:07.974Z
ORD-1042,CUST-165,Notebook,West,2,null,2026-08-18,completed,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T09:47:35.576Z
ORD-1042,CUST-165,Notebook,West,2,null,2026-08-18,completed,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T08:36:07.974Z
ORD-1059,CUST-153,Mouse,South,1,481.32,null,completed,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T08:36:07.974Z
ORD-1059,CUST-153,Mouse,South,1,481.32,null,completed,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T09:47:35.576Z
ORD-1082,CUST-172,Monitor,North,4,-45.00,2026-08-19,completed,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T09:47:35.576Z
ORD-1082,CUST-172,Monitor,North,4,-45.00,2026-08-19,completed,null,/Volumes/workspace/default/book_data/first_project_orders.csv,2026-08-21T08:36:07.974Z


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.first_project_autoload.revenue_by_region_day AS
SELECT
  region,
  order_date,
  COUNT(*) AS order_count,
  CAST(ROUND(SUM(amount), 2) AS DECIMAL(18,2)) AS total_revenue
FROM workspace.first_project_autoload.orders_silver
GROUP BY region, order_date;

SELECT COUNT(*) AS gold_rows FROM workspace.first_project_autoload.revenue_by_region_day;

gold_rows
43


In [0]:
%sql
Select * from workspace.first_project_autoload.revenue_by_region_day order by total_revenue desc;

region,order_date,order_count,total_revenue
East,2026-08-14,10,4393.46
East,2026-08-17,6,3959.78
North,2026-08-16,6,3441.01
West,2026-08-11,5,2519.06
East,2026-08-11,5,2360.32
West,2026-08-13,3,2262.23
North,2026-08-11,4,1785.07
West,2026-08-15,4,1779.33
East,2026-08-18,4,1720.63
North,2026-08-14,3,1706.73


In [0]:
%sql
SELECT
  substring_index(path, '/', -1) AS Processed_file_names
FROM cloud_files_state(
  '/Volumes/workspace/default/checkpoints/bronze/orders/'
);

Processed_file_names
first_project_orders.csv
first_project_orders_updates.csv
orders_003.csv
